# Trash robustness: Colab runner
Runtime > Change runtime type > **T4 GPU** first.

Run the cells in order. Checkpoints are saved to a shared Google Drive folder so they survive when Colab disconnects
and so Vishnu can use Süleyman's trained models for Grad-CAM.

## 1. Get the code
The repo is private, so Colab needs a GitHub token: GitHub > Settings > Developer settings > Personal access tokens >
Tokens (classic) > Generate, tick **repo**. Paste it when asked (it is not saved in the notebook).

In [ ]:
from getpass import getpass
USER = "aneesha2500"                # owner of the repo (Aneesha)
REPO = "COMP576-TrashNet"
token = getpass("GitHub token: ")
!git clone https://{token}@github.com/{USER}/{REPO}.git
%cd {REPO}
!pip install -q -r requirements.txt

## 2. Keep checkpoints on Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SHARED = "/content/drive/MyDrive/trash-robustness-shared"   # shared folder (shortcut it into My Drive)
!mkdir -p {SHARED}/checkpoints
!rm -rf checkpoints && ln -s {SHARED}/checkpoints checkpoints
!ls checkpoints

## 3. Data (every session, ~3 min)

In [ ]:
!python src/get_data.py
!python src/make_split.py          # only creates the split if splits/split.json is missing
!python src/make_masks.py --check
!python src/get_textures.py        # use --procedural if the DTD download fails
!python src/make_test_shifts.py

In [ ]:
from IPython.display import Image, display
display(Image("results/mask_check.png")); display(Image("results/shift_examples.png"))

## 4. Train (Süleyman)

In [ ]:
!python src/train.py --model resnet18 --aug none --seed 0

In [ ]:
# all runs for one model; skips runs that are already in checkpoints/
!bash scripts/run_all.sh resnet18

## 5. Grad-CAM (Vishnu)

In [ ]:
!python src/gradcam.py checkpoints/resnet18_none_s0.pt --sanity
display(Image("results/gradcam/resnet18_none_s0_examples.png"))

## 6. Summary tables and plots (Süleyman)

In [ ]:
!python src/analyze.py
!cat results/summary.md

## 7. Save results back to GitHub
Commit only `results/` (small files). Pull first so you don't overwrite your teammates' work.

In [ ]:
!git config user.name "YOUR NAME" && git config user.email "YOUR_EMAIL"
!git pull --rebase
!git add results/
!git commit -m "Add results from Colab" || echo "nothing to commit"
!git push